# 中证800 V53 Rolling 6Y Dynamic Factor LGB 实验

目标：固定 `alpha_1m` label 和 LGB regression，系统比较固定因子、动态单因子筛选、组合 IC 筛选和 forward selection。

核心假设：一套因子很难穿越所有市场阶段，但可以用过去约 6-7 个自然年的历史，保守选择下一阶段更稳的 core + satellite 因子。

实验约束：
- 只使用 `jqfactor.get_factor_values` 可取到的 JQ 因子，不使用自建 `px_` / `liq_` / `ts_` 特征。
- 每个月逐月拉因子并写缓存，避免一次性构造巨大 factor cube。
- 每次训练只用预测年份之前的数据，避免未来函数。
- 不使用 early stopping，固定 LGB 轮数，减少验证集偶然性。
- 同一个 notebook 内同时输出：fixed baseline、single-factor selector、group composite score、forward selector、forward composite score。


In [ ]:
import os
import gc
import pickle
import datetime
import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
except Exception as err:
    lgb = None
    print("lightgbm import failed:", err)

try:
    from jqdata import *
    from jqfactor import get_factor_values
except Exception as err:
    print("JoinQuant imports failed. If cached DATA_PATH exists, analysis cells can still run. err=", err)

UNIVERSE_INDEX = "000906.XSHG"
BENCHMARK_INDEX = "000906.XSHG"
OUT_DIR = "csi800_ml_v53_rolling6y_dynamic_factor_lgb_outputs"
CACHE_DIR = os.path.join(OUT_DIR, "monthly_factor_cache_jq_only_v1")
DATA_PATH = os.path.join(OUT_DIR, "csi800_jq_factor_monthly_panel.csv")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

REBUILD_DATA = True
FORCE_REFETCH_MONTHLY_CACHE = False
DATA_START_DATE = "2016-01-01"
LABEL_END_DATE = "2026-05-31"
MIN_LISTING_DAYS = 180
FACTOR_CHUNK_SIZE = 12

# User examples are 2016-2022 -> 2023, 2017-2023 -> 2024, etc.
# This is 7 calendar years inclusive. Set to 6 if you want 2017-2022 -> 2023.
TRAIN_START_YEAR_OFFSET = 7
EVAL_YEARS = [2023, 2024, 2025, 2026]

TARGET_COL = "alpha_1m"
RAW_RETURN_COL = "raw_return_1m"
BENCHMARK_RETURN_COL = "benchmark_csi800_1m"
TOP_LIST = [10, 20]

MIN_FACTOR_COVERAGE = 0.70
MIN_MONTHS_FOR_FACTOR = 24
MAX_ABS_CORR = 0.82
TARGET_FACTOR_COUNT = 24
MIN_FACTOR_COUNT = 16


FORWARD_CANDIDATE_LIMIT = 36
FORWARD_MAX_FACTORS = 20
FORWARD_MIN_GAIN = 0.0005
FORWARD_INNER_VALID_MONTHS = 12
COMPOSITE_TOP_PER_GROUP = 4

EXPORT_MODELS = False
MODEL_EXPORT_DIR = os.path.join(OUT_DIR, "exported_models")
os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)

LGB_PARAMS = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}
NUM_BOOST_ROUND = 120

print("OUT_DIR:", OUT_DIR)
print("DATA_PATH:", DATA_PATH)

In [ ]:
FACTOR_GROUPS = {
    "style": [
        "size", "non_linear_size", "beta", "market_beta", "residual_volatility", "resvol",
        "liquidity", "earnings_yield", "growth", "leverage", "momentum", "relative_momentum",
    ],
    "value_cashflow": [
        "cash_flow_to_price_ratio", "book_to_price_ratio", "sales_to_price_ratio",
        "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "cfo_to_ev",
    ],
    "quality_profit": [
        "roe_ttm", "roa_ttm", "ROAEBITTTM", "profit", "net_profit_ratio",
        "operating_profit_ratio", "profit_margin_ttm", "net_profit_to_total_operate_revenue_ttm",
        "operating_profit_to_total_profit", "adjusted_profit_to_total_profit",
        "net_operating_cash_flow_coverage", "cash_rate_of_sales",
        "goods_service_cash_to_operating_revenue_ttm", "net_operate_cash_flow_to_asset",
        "gross_profit_ttm", "operating_profit_per_share", "net_operate_cash_flow_per_share",
        "total_operating_revenue_per_share",
    ],
    "growth_balance": [
        "ACCA", "growth", "long_growth", "earnvar", "operating_revenue_growth_rate",
        "total_profit_growth_rate", "np_parent_company_owners_growth_rate", "net_profit_growth_rate",
        "net_operate_cashflow_growth_rate", "total_asset_growth_rate", "net_asset_growth_rate",
        "MLEV", "financial_leverage", "debt_to_equity_ratio", "debt_to_asset_ratio",
        "debt_to_tangible_equity_ratio", "super_quick_ratio", "net_working_capital",
    ],
    "momentum_risk": [
        "momentum", "relative_momentum", "Rank1M", "sharpe_ratio_60", "beta", "market_beta",
        "residual_volatility", "resvol", "Variance20", "Variance60", "Variance120",
    ],
    "volume_technical": [
        "VOL5", "VOL10", "VOL20", "VOL60", "VOL120", "DAVOL5", "DAVOL10", "DAVOL20",
        "turnover_volatility", "VMACD", "VOSC", "MFI14", "ATR6", "ATR14", "MACDC",
    ],
    "shape_distribution": [
        "Skewness20", "Skewness60", "Skewness120", "Kurtosis20", "Kurtosis60", "Kurtosis120",
    ],
}

GROUP_QUOTAS = {
    "style": 3,
    "value_cashflow": 5,
    "quality_profit": 5,
    "growth_balance": 3,
    "momentum_risk": 4,
    "volume_technical": 4,
    "shape_distribution": 2,
}

FIXED_JQ_BASELINE_FACTORS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit", "ACCA", "growth", "net_working_capital",
    "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20", "Kurtosis60",
]


def unique_keep_order(cols):
    seen = set()
    out = []
    for c in cols:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out


ALL_CANDIDATE_FACTORS = unique_keep_order(sum([FACTOR_GROUPS[k] for k in FACTOR_GROUPS], []) + FIXED_JQ_BASELINE_FACTORS)
FACTOR_TO_GROUP = {}
for group_name, cols in FACTOR_GROUPS.items():
    for col in cols:
        if col not in FACTOR_TO_GROUP:
            FACTOR_TO_GROUP[col] = group_name

print("candidate factor count:", len(ALL_CANDIDATE_FACTORS))
print(ALL_CANDIDATE_FACTORS)

In [ ]:
def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def first_trade_day_by_month(trade_days):
    out = []
    last_month = None
    for dt in sorted(pd.to_datetime(trade_days)):
        month = dt.strftime("%Y-%m")
        if month != last_month:
            out.append(dt)
            last_month = month
    return out


def build_month_schedule(start_date, label_end_date):
    start_ts = pd.Timestamp(start_date)
    buffer_start = (start_ts - pd.Timedelta(days=60)).strftime("%Y-%m-%d")
    all_days = list(pd.to_datetime(get_trade_days(start_date=buffer_start, end_date=label_end_date)))
    if len(all_days) == 0:
        raise ValueError("no trade days")
    first_days = first_trade_day_by_month([d for d in all_days if d >= start_ts])
    rows = []
    day_to_pos = {pd.Timestamp(d): i for i, d in enumerate(all_days)}
    for i in range(len(first_days) - 1):
        rebalance = pd.Timestamp(first_days[i])
        next_date = pd.Timestamp(first_days[i + 1])
        pos = day_to_pos.get(rebalance, None)
        if pos is None or pos <= 0:
            continue
        feature_date = pd.Timestamp(all_days[pos - 1])
        if next_date > pd.Timestamp(label_end_date):
            continue
        rows.append({
            "rebalance_date": rebalance,
            "feature_date": feature_date,
            "next_date": next_date,
        })
    return pd.DataFrame(rows)


def filter_listed_days(stock_list, feature_date, min_days):
    out = []
    for stock in stock_list:
        try:
            info = get_security_info(stock)
            if feature_date.date() - info.start_date >= datetime.timedelta(days=min_days):
                out.append(stock)
        except Exception:
            pass
    return out


def filter_st_on_date(stock_list, date):
    if len(stock_list) == 0:
        return []
    try:
        st_df = get_extras("is_st", stock_list, start_date=date, end_date=date, df=True)
        if st_df is None or st_df.empty:
            return stock_list
        s = st_df.iloc[0, :]
        return [stock for stock in stock_list if stock in s.index and not bool(s[stock])]
    except Exception as err:
        print("ST filter skipped on {} err={}".format(date, err))
        return stock_list


def fetch_factor_snapshot(stock_list, factor_cols, feature_date):
    out = pd.DataFrame(index=stock_list)
    if len(stock_list) == 0 or len(factor_cols) == 0:
        return out
    date_str = pd.Timestamp(feature_date).strftime("%Y-%m-%d")
    for factor_chunk in chunks(factor_cols, FACTOR_CHUNK_SIZE):
        try:
            factor_data = get_factor_values(stock_list, factor_chunk, end_date=date_str, count=1)
        except Exception as err:
            print("factor chunk failed", date_str, factor_chunk, err)
            factor_data = None
        for factor in factor_chunk:
            try:
                if factor_data is not None and factor in factor_data:
                    out[factor] = factor_data[factor].iloc[0, :].reindex(stock_list)
                else:
                    one = get_factor_values(stock_list, [factor], end_date=date_str, count=1)
                    if one is None or factor not in one:
                        out[factor] = np.nan
                    else:
                        out[factor] = one[factor].iloc[0, :].reindex(stock_list)
            except Exception as err:
                print("factor failed", date_str, factor, err)
                out[factor] = np.nan
        gc.collect()
    return out.reindex(index=stock_list, columns=factor_cols)


def fetch_forward_close_return(stock_list, start_date, end_date):
    out = pd.Series(index=stock_list, dtype=float)
    if len(stock_list) == 0:
        return out
    try:
        px = get_price(
            stock_list,
            start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"),
            end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"),
            frequency="daily",
            fields=["close"],
            skip_paused=False,
            fq="pre",
            panel=False,
            fill_paused=True,
        )
    except Exception as err:
        print("stock return fetch failed", start_date, end_date, err)
        return out
    if px is None or px.empty:
        return out
    px["time"] = pd.to_datetime(px["time"]).dt.normalize()
    mat = px.pivot_table(index="time", columns="code", values="close").sort_index()
    if mat.empty or len(mat) < 2:
        return out
    ret = mat.iloc[-1] / mat.iloc[0] - 1
    return ret.reindex(stock_list)


def fetch_index_close_return(index_code, start_date, end_date):
    try:
        px = get_price(
            index_code,
            start_date=pd.Timestamp(start_date).strftime("%Y-%m-%d"),
            end_date=pd.Timestamp(end_date).strftime("%Y-%m-%d"),
            frequency="daily",
            fields=["close"],
            fq="pre",
            panel=False,
        )
    except Exception as err:
        print("index return fetch failed", start_date, end_date, err)
        return np.nan
    if px is None or len(px) < 2:
        return np.nan
    close = pd.Series(px["close"]).astype(float)
    return float(close.iloc[-1] / close.iloc[0] - 1)


def build_one_month_panel(row):
    rebalance_date = pd.Timestamp(row["rebalance_date"])
    feature_date = pd.Timestamp(row["feature_date"])
    next_date = pd.Timestamp(row["next_date"])
    cache_path = os.path.join(CACHE_DIR, "panel_{}.csv".format(rebalance_date.strftime("%Y%m%d")))
    if os.path.exists(cache_path) and not FORCE_REFETCH_MONTHLY_CACHE:
        return pd.read_csv(cache_path)

    stock_list = get_index_stocks(UNIVERSE_INDEX, feature_date.strftime("%Y-%m-%d"))
    stock_list = filter_listed_days(stock_list, feature_date, MIN_LISTING_DAYS)
    stock_list = filter_st_on_date(stock_list, feature_date.strftime("%Y-%m-%d"))
    if len(stock_list) == 0:
        return pd.DataFrame()

    factor_df = fetch_factor_snapshot(stock_list, ALL_CANDIDATE_FACTORS, feature_date)
    factor_df.insert(0, "stock", factor_df.index)
    factor_df["rebalance_date"] = rebalance_date
    factor_df["feature_date"] = feature_date
    factor_df["next_date"] = next_date

    stock_ret = fetch_forward_close_return(stock_list, rebalance_date, next_date)
    bench_ret = fetch_index_close_return(BENCHMARK_INDEX, rebalance_date, next_date)
    factor_df[RAW_RETURN_COL] = factor_df["stock"].map(stock_ret)
    factor_df[BENCHMARK_RETURN_COL] = bench_ret
    factor_df[TARGET_COL] = factor_df[RAW_RETURN_COL] - factor_df[BENCHMARK_RETURN_COL]
    factor_df = factor_df.dropna(subset=[TARGET_COL]).copy()

    factor_df.to_csv(cache_path, index=False)
    print("saved", cache_path, factor_df.shape)
    gc.collect()
    return factor_df


def build_or_load_data():
    if (not REBUILD_DATA) and os.path.exists(DATA_PATH):
        return pd.read_csv(DATA_PATH)
    try:
        schedule = build_month_schedule(DATA_START_DATE, LABEL_END_DATE)
        print("schedule", schedule.shape, schedule.head(), schedule.tail())
        parts = []
        for _, row in schedule.iterrows():
            part = build_one_month_panel(row)
            if part is not None and not part.empty:
                parts.append(part)
            gc.collect()
        if len(parts) == 0:
            raise ValueError("no monthly panels built")
        df = pd.concat(parts, ignore_index=True, sort=False)
        df.to_csv(DATA_PATH, index=False)
        return df
    except NameError as err:
        if os.path.exists(DATA_PATH):
            print("JoinQuant API unavailable; loading cached DATA_PATH instead.")
            return pd.read_csv(DATA_PATH)
        raise RuntimeError("JoinQuant API unavailable and DATA_PATH not found. Run this cell in JoinQuant or upload cache. err={}".format(err))


df_raw = build_or_load_data()
print("raw loaded", df_raw.shape)
print(df_raw[["rebalance_date", "feature_date", "next_date"]].agg(["min", "max"]))

In [ ]:
def normalize_df(df):
    out = df.copy()
    for c in ["rebalance_date", "feature_date", "next_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c])
    for c in [TARGET_COL, RAW_RETURN_COL, BENCHMARK_RETURN_COL]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.dropna(subset=["stock", "rebalance_date", "next_date", TARGET_COL]).copy()
    return out


def available_factor_cols(df):
    cols = []
    for c in ALL_CANDIDATE_FACTORS:
        if c in df.columns:
            cols.append(c)
    return cols


df_all = normalize_df(df_raw)
FACTOR_COLS_AVAILABLE = available_factor_cols(df_all)
print("df_all", df_all.shape)
print("months", df_all["rebalance_date"].nunique(), df_all["rebalance_date"].min(), df_all["rebalance_date"].max())
print("available factors", len(FACTOR_COLS_AVAILABLE))
print(FACTOR_COLS_AVAILABLE)
print("missing factors", [c for c in ALL_CANDIDATE_FACTORS if c not in FACTOR_COLS_AVAILABLE])

In [ ]:
def safe_rank_ic(x, y):
    tmp = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 30:
        return np.nan
    if tmp["x"].nunique() <= 1 or tmp["y"].nunique() <= 1:
        return np.nan
    return float(tmp["x"].rank().corr(tmp["y"].rank()))


def calc_factor_metrics(train_df, factor_cols):
    rows = []
    month_rows = []
    for factor in factor_cols:
        if factor not in train_df.columns:
            continue
        coverage = float(train_df[factor].notnull().mean())
        if coverage < MIN_FACTOR_COVERAGE:
            rows.append({"factor": factor, "coverage": coverage, "eligible": False})
            continue

        ics = []
        for dt, g in train_df[["rebalance_date", TARGET_COL, factor]].groupby("rebalance_date"):
            ic = safe_rank_ic(g[factor], g[TARGET_COL])
            if not pd.isnull(ic):
                ics.append({"rebalance_date": dt, "rank_ic": ic})
                month_rows.append({"rebalance_date": dt, "factor": factor, "rank_ic": ic})
        if len(ics) < MIN_MONTHS_FOR_FACTOR:
            rows.append({"factor": factor, "coverage": coverage, "eligible": False})
            continue

        ic_df = pd.DataFrame(ics).sort_values("rebalance_date")
        ic_mean = float(ic_df["rank_ic"].mean())
        ic_std = float(ic_df["rank_ic"].std())
        direction = 1.0 if ic_mean >= 0 else -1.0
        adj_ic = ic_df["rank_ic"] * direction
        ic_ir = abs(ic_mean) / ic_std if ic_std > 0 else 0.0
        pos_month_ratio = float((adj_ic > 0).mean())
        recent_ic = float((ic_df.tail(24)["rank_ic"] * direction).mean())

        yearly = ic_df.copy()
        yearly["year"] = pd.to_datetime(yearly["rebalance_date"]).dt.year
        yearly_adj = yearly.groupby("year")["rank_ic"].mean() * direction
        worst_year_ic = float(yearly_adj.min()) if len(yearly_adj) else np.nan

        tmp = train_df[["rebalance_date", TARGET_COL, factor]].replace([np.inf, -np.inf], np.nan).dropna().copy()
        top20_vals = []
        for dt, g in tmp.groupby("rebalance_date"):
            if len(g) < 30:
                continue
            g = g.copy()
            g["factor_score"] = g[factor].rank(pct=True, method="first") * direction
            top = g.sort_values("factor_score", ascending=False).head(min(20, len(g)))
            top20_vals.append(float(top[TARGET_COL].mean()))
        top20_alpha = float(np.nanmean(top20_vals)) if len(top20_vals) else np.nan

        rows.append({
            "factor": factor,
            "group": FACTOR_TO_GROUP.get(factor, "other"),
            "coverage": coverage,
            "eligible": True,
            "direction": direction,
            "ic_mean": ic_mean,
            "abs_ic_mean": abs(ic_mean),
            "ic_ir": ic_ir,
            "pos_month_ratio": pos_month_ratio,
            "recent_ic_adj": recent_ic,
            "worst_year_ic_adj": worst_year_ic,
            "top20_alpha": top20_alpha,
            "months": len(ic_df),
        })
    metric_df = pd.DataFrame(rows)
    monthly_ic_df = pd.DataFrame(month_rows)
    if metric_df.empty:
        return metric_df, monthly_ic_df

    score_cols = ["abs_ic_mean", "ic_ir", "pos_month_ratio", "recent_ic_adj", "worst_year_ic_adj", "top20_alpha"]
    for c in score_cols:
        if c not in metric_df.columns:
            metric_df[c] = np.nan
        metric_df[c + "_rank"] = metric_df[c].rank(pct=True, na_option="bottom")
    metric_df["missing_penalty"] = (1.0 - metric_df["coverage"]).rank(pct=True, na_option="bottom")
    metric_df["factor_score"] = (
        1.20 * metric_df["abs_ic_mean_rank"] +
        1.00 * metric_df["ic_ir_rank"] +
        1.00 * metric_df["top20_alpha_rank"] +
        0.80 * metric_df["pos_month_ratio_rank"] +
        0.60 * metric_df["recent_ic_adj_rank"] +
        0.60 * metric_df["worst_year_ic_adj_rank"] -
        0.80 * metric_df["missing_penalty"]
    )
    metric_df.loc[metric_df["eligible"] != True, "factor_score"] = -999.0
    return metric_df.sort_values("factor_score", ascending=False), monthly_ic_df


def build_corr_matrix(train_df, factor_cols):
    cols = [c for c in factor_cols if c in train_df.columns]
    if len(cols) == 0:
        return pd.DataFrame()
    ranked = train_df[cols].replace([np.inf, -np.inf], np.nan).rank(pct=True)
    return ranked.corr()


def corr_ok(candidate, selected, corr_mat):
    if len(selected) == 0 or corr_mat is None or corr_mat.empty:
        return True
    if candidate not in corr_mat.index:
        return True
    vals = []
    for s in selected:
        if s in corr_mat.columns:
            vals.append(abs(corr_mat.loc[candidate, s]))
    vals = [v for v in vals if not pd.isnull(v)]
    if len(vals) == 0:
        return True
    return max(vals) <= MAX_ABS_CORR


def direction_map_from_metrics(metric_df):
    if metric_df is None or metric_df.empty or "direction" not in metric_df.columns:
        return {}
    out = {}
    for _, row in metric_df.dropna(subset=["factor"]).iterrows():
        d = row.get("direction", 1.0)
        out[row["factor"]] = 1.0 if pd.isnull(d) else float(d)
    return out


def select_dynamic_factors(train_df, candidate_cols):
    metric_df, monthly_ic_df = calc_factor_metrics(train_df, candidate_cols)
    if metric_df.empty or "eligible" not in metric_df.columns:
        return [], metric_df, monthly_ic_df
    eligible = metric_df[metric_df["eligible"] == True].copy()
    if eligible.empty:
        return [], metric_df, monthly_ic_df
    eligible = eligible.sort_values("factor_score", ascending=False)
    corr_mat = build_corr_matrix(train_df, list(eligible["factor"]))

    selected = []
    for group_name, quota in GROUP_QUOTAS.items():
        group_part = eligible[eligible["group"] == group_name].sort_values("factor_score", ascending=False)
        picked = 0
        for _, row in group_part.iterrows():
            f = row["factor"]
            if f in selected:
                continue
            if not corr_ok(f, selected, corr_mat):
                continue
            selected.append(f)
            picked += 1
            if picked >= quota or len(selected) >= TARGET_FACTOR_COUNT:
                break

    if len(selected) < MIN_FACTOR_COUNT:
        for _, row in eligible.iterrows():
            f = row["factor"]
            if f in selected:
                continue
            if not corr_ok(f, selected, corr_mat):
                continue
            selected.append(f)
            if len(selected) >= MIN_FACTOR_COUNT:
                break

    if len(selected) < TARGET_FACTOR_COUNT:
        for _, row in eligible.iterrows():
            f = row["factor"]
            if f in selected:
                continue
            if not corr_ok(f, selected, corr_mat):
                continue
            selected.append(f)
            if len(selected) >= TARGET_FACTOR_COUNT:
                break

    selected = selected[:TARGET_FACTOR_COUNT]
    return selected, metric_df, monthly_ic_df


def split_selector_fit_valid(train_df):
    months = sorted(pd.to_datetime(train_df["rebalance_date"].dropna().unique()))
    if len(months) <= FORWARD_INNER_VALID_MONTHS + 12:
        cut = max(1, int(len(months) * 0.75))
        valid_months = months[cut:]
    else:
        valid_months = months[-FORWARD_INNER_VALID_MONTHS:]
    fit = train_df[~train_df["rebalance_date"].isin(valid_months)].copy()
    valid = train_df[train_df["rebalance_date"].isin(valid_months)].copy()
    if fit.empty or valid.empty:
        return train_df.copy(), train_df.copy()
    return fit, valid


def build_directed_rank_matrix(df, factors, direction_map):
    factors = [f for f in unique_keep_order(factors) if f in df.columns]
    mat = pd.DataFrame(index=df.index)
    for f in factors:
        d = float(direction_map.get(f, 1.0))
        mat[f] = df.groupby("rebalance_date")[f].rank(pct=True, method="average") * d
    return mat.replace([np.inf, -np.inf], np.nan)


def combo_score_from_rank_matrix(rank_mat, factors):
    factors = [f for f in factors if f in rank_mat.columns]
    if len(factors) == 0:
        return pd.Series(index=rank_mat.index, dtype=float)
    return rank_mat[factors].mean(axis=1)


def eval_combo_score(df, score, topn=20):
    tmp = df[["rebalance_date", TARGET_COL]].copy()
    tmp["score"] = score
    month_ics = []
    top_vals = []
    for dt, g in tmp.replace([np.inf, -np.inf], np.nan).dropna().groupby("rebalance_date"):
        if len(g) < 30:
            continue
        ic = safe_rank_ic(g["score"], g[TARGET_COL])
        if not pd.isnull(ic):
            month_ics.append(ic)
        top = g.sort_values("score", ascending=False).head(min(topn, len(g)))
        top_vals.append(float(top[TARGET_COL].mean()))
    if len(month_ics) == 0:
        return {"combo_ic_mean": np.nan, "combo_ic_ir": np.nan, "combo_top_alpha": np.nan, "combo_objective": -999.0}
    ic_mean = float(np.nanmean(month_ics))
    ic_std = float(np.nanstd(month_ics))
    ic_ir = ic_mean / ic_std if ic_std > 0 else 0.0
    top_alpha = float(np.nanmean(top_vals)) if len(top_vals) else np.nan
    obj = ic_mean + 2.0 * (top_alpha if not pd.isnull(top_alpha) else 0.0)
    return {"combo_ic_mean": ic_mean, "combo_ic_ir": ic_ir, "combo_top_alpha": top_alpha, "combo_objective": obj}


def build_group_composite_diagnostics(train_df, metric_df):
    if metric_df is None or metric_df.empty or "eligible" not in metric_df.columns:
        return pd.DataFrame()
    eligible = metric_df[metric_df["eligible"] == True].sort_values("factor_score", ascending=False).copy()
    if eligible.empty:
        return pd.DataFrame()
    direction_map = direction_map_from_metrics(metric_df)
    rows = []
    for group_name in sorted(eligible["group"].dropna().unique()):
        group_factors = list(eligible[eligible["group"] == group_name].head(COMPOSITE_TOP_PER_GROUP)["factor"])
        if len(group_factors) == 0:
            continue
        rank_mat = build_directed_rank_matrix(train_df, group_factors, direction_map)
        score = combo_score_from_rank_matrix(rank_mat, group_factors)
        stats = eval_combo_score(train_df, score, topn=20)
        stats.update({
            "group": group_name,
            "factor_count": len(group_factors),
            "factors": ",".join(group_factors),
        })
        rows.append(stats)
    return pd.DataFrame(rows).sort_values("combo_objective", ascending=False) if rows else pd.DataFrame()


def select_group_composite_factors(train_df, metric_df):
    if metric_df is None or metric_df.empty or "eligible" not in metric_df.columns:
        return []
    eligible = metric_df[metric_df["eligible"] == True].sort_values("factor_score", ascending=False).copy()
    if eligible.empty:
        return []
    selected = []
    corr_mat = build_corr_matrix(train_df, list(eligible["factor"]))
    for group_name in sorted(eligible["group"].dropna().unique()):
        group_part = eligible[eligible["group"] == group_name].head(COMPOSITE_TOP_PER_GROUP)
        for _, row in group_part.iterrows():
            f = row["factor"]
            if f not in selected and corr_ok(f, selected, corr_mat):
                selected.append(f)
    return selected[:TARGET_FACTOR_COUNT]


def select_forward_factors(train_df, candidate_cols):
    fit_df, valid_df = split_selector_fit_valid(train_df)
    metric_fit, _ = calc_factor_metrics(fit_df, candidate_cols)
    if metric_fit.empty or "eligible" not in metric_fit.columns:
        metric_full, _ = calc_factor_metrics(train_df, candidate_cols)
        return [], pd.DataFrame(), metric_full
    eligible = metric_fit[metric_fit["eligible"] == True].sort_values("factor_score", ascending=False).copy()
    if eligible.empty:
        metric_full, _ = calc_factor_metrics(train_df, candidate_cols)
        return [], pd.DataFrame(), metric_full
    candidate_pool = list(eligible.head(FORWARD_CANDIDATE_LIMIT)["factor"])
    if len(candidate_pool) == 0:
        metric_full, _ = calc_factor_metrics(train_df, candidate_cols)
        return [], pd.DataFrame(), metric_full
    direction_map = direction_map_from_metrics(metric_fit)
    corr_mat = build_corr_matrix(fit_df, candidate_pool)
    rank_mat_valid = build_directed_rank_matrix(valid_df, candidate_pool, direction_map)

    selected = []
    trace_rows = []
    best_obj = -999.0
    for step in range(1, FORWARD_MAX_FACTORS + 1):
        best_factor = None
        best_stats = None
        for f in candidate_pool:
            if f in selected:
                continue
            if not corr_ok(f, selected, corr_mat):
                continue
            trial = selected + [f]
            score = combo_score_from_rank_matrix(rank_mat_valid, trial)
            stats = eval_combo_score(valid_df, score, topn=20)
            obj = stats["combo_objective"]
            if best_stats is None or obj > best_stats["combo_objective"]:
                best_factor = f
                best_stats = stats
        if best_factor is None or best_stats is None:
            break
        gain = best_stats["combo_objective"] - best_obj
        if step > 1 and gain < FORWARD_MIN_GAIN:
            break
        selected.append(best_factor)
        best_obj = best_stats["combo_objective"]
        row = dict(best_stats)
        row.update({"step": step, "factor_added": best_factor, "selected_factors": ",".join(selected), "gain": gain})
        trace_rows.append(row)

    # Recompute directions on the full train window for final train/test scoring only after selection is fixed.
    metric_full, _ = calc_factor_metrics(train_df, candidate_cols)
    return selected, pd.DataFrame(trace_rows), metric_full

In [ ]:
def prepare_xy(df, feature_cols, target_col):
    data = df.replace([np.inf, -np.inf], np.nan).copy()
    X = data[feature_cols].copy()
    fill_values = X.median().to_dict()
    X = X.fillna(pd.Series(fill_values)).fillna(0.0)
    y = pd.to_numeric(data[target_col], errors="coerce")
    valid = y.notnull()
    return X.loc[valid], y.loc[valid], fill_values


def train_lgb_model(train_df, feature_cols, target_col):
    if lgb is None:
        raise RuntimeError("lightgbm is not available")
    X, y, fill_values = prepare_xy(train_df, feature_cols, target_col)
    dtrain = lgb.Dataset(X, label=y, feature_name=feature_cols)
    model = lgb.train(dict(LGB_PARAMS), dtrain, num_boost_round=NUM_BOOST_ROUND)
    imp = pd.DataFrame({
        "feature": feature_cols,
        "gain_importance": model.feature_importance(importance_type="gain"),
        "split_importance": model.feature_importance(importance_type="split"),
    })
    return model, fill_values, imp, len(X)


def score_with_model(model, fill_values, df, feature_cols):
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(pd.Series(fill_values)).fillna(0.0)
    return np.asarray(model.predict(X[feature_cols])).reshape(-1)


def max_drawdown(ret_series):
    if len(ret_series) == 0:
        return np.nan
    nav = (1.0 + pd.Series(ret_series).fillna(0.0)).cumprod()
    peak = nav.cummax()
    dd = nav / peak - 1.0
    return float(dd.min())


def evaluate_topn(score_df, strategy_name, topn):
    rows = []
    for dt, g in score_df.groupby("rebalance_date"):
        g = g.sort_values("score", ascending=False)
        top = g.head(min(topn, len(g)))
        raw_ret = float(top[RAW_RETURN_COL].mean()) if len(top) else np.nan
        bench_ret = float(top[BENCHMARK_RETURN_COL].iloc[0]) if len(top) else np.nan
        alpha = raw_ret - bench_ret
        rows.append({
            "strategy_name": strategy_name,
            "portfolio_profile": "top{}".format(topn),
            "rebalance_date": dt,
            "raw_return_1m": raw_ret,
            "benchmark_csi800_1m": bench_ret,
            "alpha_1m": alpha,
            "target_count": int(len(top)),
            "targets": ",".join(list(top["stock"])),
        })
    monthly = pd.DataFrame(rows).sort_values("rebalance_date")
    if monthly.empty:
        return monthly, {}
    cum_ret = float((1.0 + monthly["raw_return_1m"].fillna(0.0)).prod() - 1.0)
    cum_bench = float((1.0 + monthly["benchmark_csi800_1m"].fillna(0.0)).prod() - 1.0)
    cum_excess = float((1.0 + cum_ret) / (1.0 + cum_bench) - 1.0) if (1.0 + cum_bench) != 0 else np.nan
    summary = {
        "strategy_name": strategy_name,
        "portfolio_profile": "top{}".format(topn),
        "months": int(len(monthly)),
        "cum_ret": cum_ret,
        "cum_csi800": cum_bench,
        "cum_excess_csi800": cum_excess,
        "mean_monthly_excess": float(monthly["alpha_1m"].mean()),
        "win_rate": float((monthly["alpha_1m"] > 0).mean()),
        "max_drawdown": max_drawdown(monthly["raw_return_1m"]),
    }
    return monthly, summary


def build_windows(df):
    rows = []
    for test_year in EVAL_YEARS:
        train_start_year = test_year - TRAIN_START_YEAR_OFFSET
        train_start = pd.Timestamp("{}-01-01".format(train_start_year))
        train_end = pd.Timestamp("{}-12-31".format(test_year - 1))
        test_start = pd.Timestamp("{}-01-01".format(test_year))
        test_end = pd.Timestamp("{}-12-31".format(test_year))
        train_df = df[(df["rebalance_date"] >= train_start) & (df["rebalance_date"] <= train_end)].copy()
        test_df = df[(df["rebalance_date"] >= test_start) & (df["rebalance_date"] <= test_end)].copy()
        if train_df.empty or test_df.empty:
            continue
        rows.append({
            "test_year": int(test_year),
            "window_name": "train{}_{}_test{}".format(train_start_year, test_year - 1, test_year),
            "train_start": train_start,
            "train_end": train_end,
            "test_start": test_start,
            "test_end": test_end,
        })
    return pd.DataFrame(rows)


windows_df = build_windows(df_all)
print(windows_df)

In [ ]:
def run_one_model(train_df, test_df, feature_cols, strategy_name, window_row):
    feature_cols = [c for c in unique_keep_order(feature_cols) if c in train_df.columns and c in test_df.columns]
    if len(feature_cols) == 0:
        return pd.DataFrame(), pd.DataFrame(), {}
    model, fill_values, importance, train_rows = train_lgb_model(train_df, feature_cols, TARGET_COL)
    score_df = test_df[["stock", "rebalance_date", "feature_date", "next_date", RAW_RETURN_COL, BENCHMARK_RETURN_COL, TARGET_COL]].copy()
    score_df["score"] = score_with_model(model, fill_values, test_df, feature_cols)
    score_df["strategy_name"] = strategy_name
    score_df["window_name"] = window_row["window_name"]
    score_df["feature_count"] = len(feature_cols)

    importance["strategy_name"] = strategy_name
    importance["window_name"] = window_row["window_name"]
    importance["test_year"] = int(window_row["test_year"])

    meta = {
        "strategy_name": strategy_name,
        "window_name": window_row["window_name"],
        "test_year": int(window_row["test_year"]),
        "train_start": window_row["train_start"],
        "train_end": window_row["train_end"],
        "test_start": window_row["test_start"],
        "test_end": window_row["test_end"],
        "train_rows": int(train_rows),
        "test_rows": int(len(test_df)),
        "feature_count": int(len(feature_cols)),
        "features": ",".join(feature_cols),
        "score_type": "lgb",
    }

    if EXPORT_MODELS:
        bundle = {
            "objective": "v53_rolling6y_dynamic_factor_lgb",
            "strategy_name": strategy_name,
            "window_name": window_row["window_name"],
            "target_col": TARGET_COL,
            "base_model": model,
            "base_feature_cols": list(feature_cols),
            "base_fill_values": dict(fill_values),
            "base_params": dict(LGB_PARAMS),
            "fixed_iter": int(NUM_BOOST_ROUND),
            "stock_num": 10,
            "benchmark": BENCHMARK_INDEX,
        }
        pkl_path = os.path.join(MODEL_EXPORT_DIR, "model_{}_{}.pkl".format(strategy_name, window_row["window_name"]))
        with open(pkl_path, "wb") as f:
            pickle.dump(bundle, f, protocol=2)
        meta["model_path"] = pkl_path
    return score_df, importance, meta


def run_composite_strategy(test_df, feature_cols, metric_df, strategy_name, window_row):
    feature_cols = [c for c in unique_keep_order(feature_cols) if c in test_df.columns]
    if len(feature_cols) == 0:
        return pd.DataFrame(), {}
    direction_map = direction_map_from_metrics(metric_df)
    rank_mat = build_directed_rank_matrix(test_df, feature_cols, direction_map)
    score = combo_score_from_rank_matrix(rank_mat, feature_cols)
    score_df = test_df[["stock", "rebalance_date", "feature_date", "next_date", RAW_RETURN_COL, BENCHMARK_RETURN_COL, TARGET_COL]].copy()
    score_df["score"] = score
    score_df["strategy_name"] = strategy_name
    score_df["window_name"] = window_row["window_name"]
    score_df["feature_count"] = len(feature_cols)
    meta = {
        "strategy_name": strategy_name,
        "window_name": window_row["window_name"],
        "test_year": int(window_row["test_year"]),
        "train_start": window_row["train_start"],
        "train_end": window_row["train_end"],
        "test_start": window_row["test_start"],
        "test_end": window_row["test_end"],
        "train_rows": np.nan,
        "test_rows": int(len(test_df)),
        "feature_count": int(len(feature_cols)),
        "features": ",".join(feature_cols),
        "score_type": "directed_rank_composite",
    }
    return score_df, meta


score_parts = []
importance_parts = []
meta_rows = []
factor_metric_parts = []
monthly_ic_parts = []
selected_rows = []
monthly_parts = []
summary_rows = []
group_composite_parts = []
forward_trace_parts = []

for _, win in windows_df.iterrows():
    print("running", win["window_name"])
    train_df = df_all[(df_all["rebalance_date"] >= win["train_start"]) & (df_all["rebalance_date"] <= win["train_end"])].copy()
    test_df = df_all[(df_all["rebalance_date"] >= win["test_start"]) & (df_all["rebalance_date"] <= win["test_end"])].copy()

    selected_single, metric_df, monthly_ic_df = select_dynamic_factors(train_df, FACTOR_COLS_AVAILABLE)
    selected_group = select_group_composite_factors(train_df, metric_df)
    selected_forward, forward_trace_df, forward_metric_df = select_forward_factors(train_df, FACTOR_COLS_AVAILABLE)
    print("single selected", len(selected_single), selected_single)
    print("group selected", len(selected_group), selected_group)
    print("forward selected", len(selected_forward), selected_forward)

    group_diag_df = build_group_composite_diagnostics(train_df, metric_df)
    if not group_diag_df.empty:
        group_diag_df["window_name"] = win["window_name"]
        group_diag_df["test_year"] = int(win["test_year"])
        group_composite_parts.append(group_diag_df)
    if not forward_trace_df.empty:
        forward_trace_df["window_name"] = win["window_name"]
        forward_trace_df["test_year"] = int(win["test_year"])
        forward_trace_parts.append(forward_trace_df)

    if not metric_df.empty:
        metric_df["window_name"] = win["window_name"]
        metric_df["test_year"] = int(win["test_year"])
        metric_df["metric_source"] = "full_train"
        factor_metric_parts.append(metric_df)
    if not forward_metric_df.empty:
        fm = forward_metric_df.copy()
        fm["window_name"] = win["window_name"]
        fm["test_year"] = int(win["test_year"])
        fm["metric_source"] = "forward_full_train_after_selection"
        factor_metric_parts.append(fm)
    if not monthly_ic_df.empty:
        monthly_ic_df["window_name"] = win["window_name"]
        monthly_ic_df["test_year"] = int(win["test_year"])
        monthly_ic_parts.append(monthly_ic_df)

    selected_specs = [
        ("single_factor_selector", selected_single),
        ("group_composite_selector", selected_group),
        ("forward_combo_selector", selected_forward),
        ("fixed_jq_baseline", [c for c in FIXED_JQ_BASELINE_FACTORS if c in FACTOR_COLS_AVAILABLE]),
    ]
    for selector_name, features in selected_specs:
        selected_rows.append({
            "selector_name": selector_name,
            "window_name": win["window_name"],
            "test_year": int(win["test_year"]),
            "feature_count": len(features),
            "features": ",".join(features),
        })

    model_specs = [
        ("single_factor_selected_lgb", selected_single),
        ("group_composite_selected_lgb", selected_group),
        ("forward_selected_lgb", selected_forward),
        ("fixed_jq_baseline_lgb", [c for c in FIXED_JQ_BASELINE_FACTORS if c in FACTOR_COLS_AVAILABLE]),
    ]
    composite_specs = [
        ("single_factor_selected_composite", selected_single, metric_df),
        ("group_composite_score", selected_group, metric_df),
        ("forward_selected_composite", selected_forward, forward_metric_df),
        ("fixed_jq_baseline_composite", [c for c in FIXED_JQ_BASELINE_FACTORS if c in FACTOR_COLS_AVAILABLE], metric_df),
    ]

    for strategy_name, features in model_specs:
        score_df_part, imp_df, meta = run_one_model(train_df, test_df, features, strategy_name, win)
        if score_df_part.empty:
            continue
        score_parts.append(score_df_part)
        importance_parts.append(imp_df)
        meta_rows.append(meta)
        for topn in TOP_LIST:
            m, s = evaluate_topn(score_df_part, strategy_name, topn)
            if not m.empty:
                m["window_name"] = win["window_name"]
                m["test_year"] = int(win["test_year"])
                monthly_parts.append(m)
                s["window_name"] = win["window_name"]
                s["test_year"] = int(win["test_year"])
                summary_rows.append(s)

    for strategy_name, features, metric_source in composite_specs:
        score_df_part, meta = run_composite_strategy(test_df, features, metric_source, strategy_name, win)
        if score_df_part.empty:
            continue
        score_parts.append(score_df_part)
        meta_rows.append(meta)
        for topn in TOP_LIST:
            m, s = evaluate_topn(score_df_part, strategy_name, topn)
            if not m.empty:
                m["window_name"] = win["window_name"]
                m["test_year"] = int(win["test_year"])
                monthly_parts.append(m)
                s["window_name"] = win["window_name"]
                s["test_year"] = int(win["test_year"])
                summary_rows.append(s)
    gc.collect()

score_df = pd.concat(score_parts, ignore_index=True, sort=False) if score_parts else pd.DataFrame()
importance_df = pd.concat(importance_parts, ignore_index=True, sort=False) if importance_parts else pd.DataFrame()
model_meta_df = pd.DataFrame(meta_rows)
factor_metrics_df = pd.concat(factor_metric_parts, ignore_index=True, sort=False) if factor_metric_parts else pd.DataFrame()
monthly_factor_ic_df = pd.concat(monthly_ic_parts, ignore_index=True, sort=False) if monthly_ic_parts else pd.DataFrame()
selected_factors_df = pd.DataFrame(selected_rows)
monthly_df = pd.concat(monthly_parts, ignore_index=True, sort=False) if monthly_parts else pd.DataFrame()
summary_df = pd.DataFrame(summary_rows).sort_values(["test_year", "strategy_name", "portfolio_profile"]) if summary_rows else pd.DataFrame()
group_composite_df = pd.concat(group_composite_parts, ignore_index=True, sort=False) if group_composite_parts else pd.DataFrame()
forward_trace_df = pd.concat(forward_trace_parts, ignore_index=True, sort=False) if forward_trace_parts else pd.DataFrame()

print("summary")
print(summary_df)

In [ ]:
score_df.to_csv(os.path.join(OUT_DIR, "v53_scores.csv"), index=False)
importance_df.to_csv(os.path.join(OUT_DIR, "v53_importance.csv"), index=False)
model_meta_df.to_csv(os.path.join(OUT_DIR, "v53_model_meta.csv"), index=False)
factor_metrics_df.to_csv(os.path.join(OUT_DIR, "v53_factor_metrics_by_window.csv"), index=False)
monthly_factor_ic_df.to_csv(os.path.join(OUT_DIR, "v53_monthly_factor_ic.csv"), index=False)
selected_factors_df.to_csv(os.path.join(OUT_DIR, "v53_selected_factors.csv"), index=False)
monthly_df.to_csv(os.path.join(OUT_DIR, "v53_monthly.csv"), index=False)
summary_df.to_csv(os.path.join(OUT_DIR, "v53_summary.csv"), index=False)
group_composite_df.to_csv(os.path.join(OUT_DIR, "v53_group_composite_diagnostics.csv"), index=False)
forward_trace_df.to_csv(os.path.join(OUT_DIR, "v53_forward_selection_trace.csv"), index=False)

print("saved outputs to", OUT_DIR)
print("selected factors:")
print(selected_factors_df)
print("group composite diagnostics:")
print(group_composite_df.head(30) if not group_composite_df.empty else group_composite_df)
print("forward trace:")
print(forward_trace_df.head(40) if not forward_trace_df.empty else forward_trace_df)
print("summary:")
print(summary_df)

## 结论填写区

运行后重点看：

1. `single_factor_selected_lgb` 是否优于 `fixed_jq_baseline_lgb`，判断单因子稳定性筛选是否有价值。
2. `group_composite_score` / `single_factor_selected_composite` 是否本身有 IC 和收益，判断多因子组合是否不依赖 LGB 也有效。
3. `forward_selected_lgb` 和 `forward_selected_composite` 是否在 2023 这种困难年份少崩，判断弱因子互补是否存在。
4. `v53_group_composite_diagnostics.csv` 看哪些因子组的 composite IC 稳定，避免只看单因子。
5. `v53_forward_selection_trace.csv` 看 greedy 每一步是否还有边际增益；如果第 5-8 步后增益快速消失，说明不应选太多因子。
6. 如果 LGB 赢但 composite 全输，说明可能依赖非线性交互，需要更严格 OOS；如果 composite 也赢，说明因子组合本身更可信。
